# Make your own (perceptually uniform) colormap

A colormap is a function from numbers to colors. Most of them are bad, and bad ones do real
damage: they invent boundaries that are not in the data, hide differences that are, and fall
apart in grayscale or for a colorblind reader.

This notebook has two halves:

1. **Design** your colormap in the browser tool `Colormap Designer.html` (double-click it — it
   opens in any browser, nothing to install). You draw a path through a perceptual color space,
   drag the points around, and it grades you out of 100.
2. **Use and verify** it here, on real arrays, with the same math the grader uses.

> **Why not `viscm`?** `viscm edit` is the research tool this activity is based on, but it needs a
> desktop Qt window and does not run inside a notebook. Everything it shows you is reproduced in
> the browser tool and below, with no installs.

Run this cell first.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("ready — numpy", np.__version__)

## 1. Load the colormap you designed

In the browser tool, click **Evaluate my colormap**, then **Python / matplotlib**, then **Copy**.
Paste over the `MY_COLORS` list below.

You can paste either the full 256-color list the tool gives you, or just a handful of anchor
colors — this builds a smooth colormap from however many you provide.

> **One caveat on the numbers.** Hex codes are 8-bit, so a pasted 256-color list has been rounded
> to 1/255 per channel. The "wobble" figure below is a *second* derivative, which amplifies that
> rounding badly — 8-bit viridis measures a wobble of 0.27 against 0.0002 for the real thing. So
> if you paste all 256, expect the wobble number here to look far worse than the browser tool
> reported, and ignore it. Every other number is fine. Paste a handful of **anchor** colors
> instead if you want a like-for-like wobble.

If you used the **Download .jscm** button instead, use `MY_COLORS = load_jscm("my_cmap.jscm")`.

In [ ]:
# ------------------------------------------------------------------
# REPLACE THIS with your own colors from the browser tool.
# (This starter is a reasonable first attempt — improve on it.)
# ------------------------------------------------------------------
MY_COLORS = [
    "#221033", "#333f7a", "#2c7395", "#41a68c", "#a2cf6a", "#f2f0a8",
]

def load_jscm(path):
    """Read the 256 colors out of a .jscm file (the viscm / matplotlib format)."""
    s = json.load(open(path))["colors"]
    return ["#" + s[i:i + 6] for i in range(0, len(s), 6)]

my_cmap = LinearSegmentedColormap.from_list("my_cmap", MY_COLORS, N=256)

# The 256 RGB values this colormap actually produces. Everything below measures these.
my_rgb = my_cmap(np.linspace(0, 1, 256))[:, :3]

fig, ax = plt.subplots(figsize=(7, 0.6))
ax.imshow(my_rgb[None, :, :], aspect="auto", extent=[0, 1, 0, 1])
ax.set(yticks=[], xticks=[0, 1], title="my_cmap")
ax.grid(False)
plt.show()

## 2. Look at it on data

A colormap can only be judged on data. These three arrays are the same ones the browser tool
uses, and each one exposes a different failure mode:

- a **smooth surface**, where a bad colormap shows ridges that are not there
- a **coarse grid**, where you need to read individual cells against the colorbar
- a **fine oscillating pattern**, where a bad colormap loses whole cycles

In [ ]:
def smooth_surface(w=240, h=150, seed=3):
    """Low-frequency noise plus a broad peak — like elevation or a density estimate."""
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:h, 0:w]
    z = np.zeros((h, w))
    for octave in range(5):
        n = 4 * 2 ** octave
        coarse = rng.random((n + 1, n + 1))
        yi = np.linspace(0, n, h)
        xi = np.linspace(0, n, w)
        y0, x0 = np.floor(yi).astype(int), np.floor(xi).astype(int)
        fy, fx = (yi - y0)[:, None], (xi - x0)[None, :]
        sy, sx = fy * fy * (3 - 2 * fy), fx * fx * (3 - 2 * fx)
        y1, x1 = np.minimum(y0 + 1, n), np.minimum(x0 + 1, n)
        top = coarse[np.ix_(y0, x0)] * (1 - sx) + coarse[np.ix_(y0, x1)] * sx
        bot = coarse[np.ix_(y1, x0)] * (1 - sx) + coarse[np.ix_(y1, x1)] * sx
        z += (top * (1 - sy) + bot * sy) / 2 ** octave
    z += 1.5 * np.exp(-(((x - w * 0.42) / (w * 0.30)) ** 2 + ((y - h * 0.46) / (h * 0.30)) ** 2))
    return (z - z.min()) / np.ptp(z)


def coarse_grid(w=26, h=20, seed=7):
    """A small number of noisy measurements — read cell by cell."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:h, 0:w]
    u, v = xx / w, yy / h
    z = np.zeros((h, w))
    for cx, cy, sx, sy, amp in [(0.34, 0.34, 0.30, 0.09, 1.0), (0.66, 0.66, 0.26, 0.08, 0.85)]:
        dx, dy = u - cx, v - cy
        rot = -0.7
        rx = dx * np.cos(rot) - dy * np.sin(rot)
        ry = dx * np.sin(rot) + dy * np.cos(rot)
        z += amp * np.exp(-(rx ** 2 / sx ** 2 + ry ** 2 / sy ** 2))
    z += 0.22 * rng.random((h, w))
    return (z - z.min()) / np.ptp(z)


def interference(w=250, h=150):
    """Two-source interference — lots of closely spaced cycles."""
    y, x = np.mgrid[0:h, 0:w]
    u, v = (x / w - 0.5) * 8, (y / h - 0.5) * 5
    z = (np.cos(np.hypot(u + 1.6, v) * 4.2) + np.cos(np.hypot(u - 1.6, v) * 4.2)
         + 0.35 * np.cos((u + v) * 2.2))
    return (z - z.min()) / np.ptp(z)


DATASETS = [("smooth surface", smooth_surface()),
            ("coarse grid", coarse_grid()),
            ("fine oscillation", interference())]


def show_on_data(cmap, title):
    fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
    for ax, (name, z) in zip(axes, DATASETS):
        im = ax.imshow(z, cmap=cmap, interpolation="nearest")
        ax.set(title=name, xticks=[], yticks=[])
        ax.grid(False)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(title, y=1.04)
    plt.show()


show_on_data(my_cmap, "your colormap")

## 3. Measure it

Now the part that turns opinion into numbers. To ask "do equal steps in the data *look* like equal
steps?" you need a color space where distance means perceived difference. RGB is not one: the
distance from black to blue is not perceptually the same as black to yellow.

**CAM02-UCS** is one, and it is what `viscm` and the makers of *viridis* used. The cell below
converts sRGB into it. You can skip the details — it is a direct port of the published
CIECAM02 model, verified against the standard test values in the docstring.

In [ ]:
# ============================================================================
# CAM02-UCS conversion + colorblindness simulation. Utility code — skim it.
# ============================================================================
_M_RGB2XYZ = np.array([[0.4124564, 0.3575761, 0.1804375],
                       [0.2126729, 0.7151522, 0.0721750],
                       [0.0193339, 0.1191920, 0.9503041]])
_MCAT02 = np.array([[0.7328, 0.4296, -0.1624],
                    [-0.7036, 1.6975, 0.0061],
                    [0.0030, 0.0136, 0.9834]])
_MHPE = np.array([[0.38971, 0.68898, -0.07868],
                  [-0.22981, 1.18340, 0.04641],
                  [0.0, 0.0, 1.0]])

# Viewing conditions: D65 white, dim-ish average surround. Same as viscm.
_XYZ_W = np.array([95.047, 100.0, 108.883])
_Y_B, _L_A, _F, _C_SUR, _N_C = 20.0, 64 / np.pi / 5, 1.0, 0.69, 1.0

_rgb_w = _MCAT02 @ _XYZ_W
_D = np.clip(_F * (1 - (1 / 3.6) * np.exp((-_L_A - 42) / 92)), 0, 1)
_D_RGB = _D * _XYZ_W[1] / _rgb_w + 1 - _D
_k = 1 / (5 * _L_A + 1)
_F_L = 0.2 * _k ** 4 * (5 * _L_A) + 0.1 * (1 - _k ** 4) ** 2 * np.cbrt(5 * _L_A)
_n = _Y_B / _XYZ_W[1]
_z = 1.48 + np.sqrt(_n)
_N_BB = 0.725 * (1 / _n) ** 0.2
_M_FWD = _MHPE @ np.linalg.inv(_MCAT02)


def _adapt(r):
    t = (_F_L * np.abs(r) / 100) ** 0.42
    return np.sign(r) * 400 * t / (27.13 + t) + 0.1


_rgb_aw = _adapt(_M_FWD @ (_D_RGB * _rgb_w))
_A_W = (2 * _rgb_aw[0] + _rgb_aw[1] + _rgb_aw[2] / 20 - 0.305) * _N_BB


def srgb_to_linear(c):
    c = np.asarray(c, float)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)


def linear_to_srgb(c):
    c = np.clip(np.asarray(c, float), 0, 1)
    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055)


def rgb_to_Jab(rgb):
    """sRGB in 0..1, shape (..., 3)  ->  CAM02-UCS (J', a', b').

    Verified against the published CIECAM02 worked example: XYZ (19.01, 20.00, 21.78)
    under L_A = 318.31 gives J = 41.73, M = 0.1088, h = 219.0.
    """
    rgb = np.asarray(rgb, float)
    xyz = srgb_to_linear(rgb) @ _M_RGB2XYZ.T * 100
    rgb_a = _adapt((xyz @ _MCAT02.T) * _D_RGB @ _M_FWD.T)
    R, G, B = rgb_a[..., 0], rgb_a[..., 1], rgb_a[..., 2]

    a = R - 12 * G / 11 + B / 11
    b = (R + G - 2 * B) / 9
    h = np.arctan2(b, a)
    e_t = 0.25 * (np.cos(h + 2) + 3.8)

    A = np.clip((2 * R + G + B / 20 - 0.305) * _N_BB, 0, None)
    J = 100 * (A / _A_W) ** (_C_SUR * _z)

    t = ((50000 / 13) * _N_C * _N_BB * e_t * np.hypot(a, b)
         / (R + G + 21 * B / 20))
    C = np.clip(t, 0, None) ** 0.9 * np.sqrt(J / 100) * (1.64 - 0.29 ** _n) ** 0.73
    M = C * _F_L ** 0.25

    J_p = (1 + 100 * 0.007) * J / (1 + 0.007 * J)      # CAM02-UCS lightness
    M_p = np.log1p(0.0228 * M) / 0.0228                # ... and colorfulness
    return np.stack([J_p, M_p * np.cos(h), M_p * np.sin(h)], axis=-1)


# Machado, Oliveira & Fernandes (2009) dichromacy matrices, applied in linear RGB.
_CVD = {
    "deuteranopia": np.array([[0.367322, 0.860646, -0.227968],
                              [0.280085, 0.672501, 0.047413],
                              [-0.011820, 0.042940, 0.968881]]),
    "protanopia":   np.array([[0.152286, 1.052583, -0.204868],
                              [0.114503, 0.786281, 0.099216],
                              [-0.003882, -0.048116, 1.051998]]),
    "tritanopia":   np.array([[1.255528, -0.076749, -0.178779],
                              [-0.078411, 0.930809, 0.147602],
                              [0.004733, 0.691367, 0.303900]]),
}


def simulate_cvd(rgb, kind, severity=1.0):
    """How `rgb` looks to someone with this color vision deficiency."""
    M = (1 - severity) * np.eye(3) + severity * _CVD[kind]
    return linear_to_srgb(srgb_to_linear(rgb) @ M.T)


def to_grayscale(rgb):
    """What a black-and-white printer does."""
    y = srgb_to_linear(rgb) @ np.array([0.2126729, 0.7151522, 0.0721750])
    return np.repeat(linear_to_srgb(y)[..., None], 3, axis=-1)


print("CAM02-UCS ready.  white ->", rgb_to_Jab([1, 1, 1]).round(2),
      " black ->", rgb_to_Jab([0, 0, 0]).round(2))

### The two plots that matter

**Perceptual deltas.** The distance in CAM02-UCS between neighboring colors — the rate of change
along your colormap. If this line is flat, equal steps in your data are equal steps to the eye.
If it spikes, your colormap exaggerates that part of the range; if it dips to zero, it hides it.

Two separate things can go wrong here, and the tool scores them separately. The curve can sit at
the wrong **level** in places (some stretches change faster than others), and it can **wobble** —
keep changing slope from step to step. That wobble is the derivative of this curve, i.e. the
second derivative of perceived color along your path, and it is the thing that makes a colormap
look gritty rather than merely uneven.

**Lightness.** For a sequential map this must climb steadily and never turn back. That is what
makes it survive a black-and-white printout and read unambiguously as "more".

In [ ]:
def diagnose(rgb, name="my_cmap"):
    jab = rgb_to_Jab(rgb)
    steps = np.linalg.norm(np.diff(jab, axis=0), axis=1)
    J = jab[:, 0]
    dJ = np.diff(J)

    cv = steps.std() / steps.mean()
    monotonic = abs(dJ.sum()) / np.abs(dJ).sum()
    # Derivative of the deltas curve = second derivative of perceived color.
    # Normalized so it reads as "average step-to-step wobble, as a fraction of
    # the flat ideal line's height".
    second_deriv = np.abs(np.diff(steps)).mean() / steps.mean()

    fig, axes = plt.subplots(1, 2, figsize=(11, 2.9))
    x = np.linspace(0, 1, len(steps))
    axes[0].plot(x, steps, color="0.15", lw=1.2)
    axes[0].axhline(steps.mean(), ls="--", lw=1.2, color="#1b6350",
                    label=f"mean {steps.mean():.2f} — the ideal flat line")
    axes[0].set(xlabel="position along the colormap", ylabel="ΔE per step", ylim=0,
                title=f"perceptual deltas   (level varies {cv:.0%}, wobble {second_deriv:.1%}/step)")
    axes[0].legend(fontsize=8, frameon=False)

    axes[1].plot(np.linspace(0, 1, len(J)), J, color="0.15", lw=1.2)
    axes[1].set(xlabel="position along the colormap", ylabel="lightness J'", ylim=(0, 100),
                title=f"lightness   ({monotonic:.0%} monotonic, span {np.ptp(J):.0f})")
    fig.suptitle(name, y=1.04)
    plt.show()

    print(f"deltas wobble       {second_deriv:6.2%}   per step — the second derivative of perception")
    print(f"                              (viridis 0.02%, jet 3.3%, prism 48%; lower is better)")
    print(f"step size varies by {cv:6.1%}   (viridis ≈ 1%, jet ≈ 43% — lower is better)")
    print(f"total perceptual ΔE {steps.sum():6.1f}   (viridis ≈ 122, pure grayscale ≈ 100;")
    print(f"                              this is the map's resolving power — bigger is better)")
    print(f"lightness monotonic {monotonic:6.1%}   (want ~100% for a sequential map)")
    print(f"lightness span      {np.ptp(J):6.1f}   (want 60+ so the ends are clearly different)")
    return cv, monotonic


diagnose(my_rgb, "your colormap");

### How it degrades

Same colormap, seen five other ways. Look for two things: does the **grayscale** row still go
steadily from dark to light, and do any two clearly-different values in the color row become the
**same** color in the colorblind rows?

In [ ]:
def show_degradations(rgb, name="my_cmap"):
    rows = [
        ("your colormap", rgb),
        ("black & white print", to_grayscale(rgb)),
        ("moderate deuteranomaly (~50%)", simulate_cvd(rgb, "deuteranopia", 0.5)),
        ("deuteranopia (~6% of men)", simulate_cvd(rgb, "deuteranopia")),
        ("protanopia (~2% of men)", simulate_cvd(rgb, "protanopia")),
        ("tritanopia (rare)", simulate_cvd(rgb, "tritanopia")),
    ]
    fig, axes = plt.subplots(len(rows), 1, figsize=(7.5, 0.42 * len(rows) + 1.1))
    for ax, (label, cols) in zip(axes, rows):
        ax.imshow(np.clip(cols, 0, 1)[None, :, :], aspect="auto")
        ax.set(xticks=[], yticks=[], ylabel="")
        ax.set_title(label, fontsize=8, loc="left", pad=2)
        ax.grid(False)
    fig.suptitle(name, y=1.0)
    plt.tight_layout()
    plt.show()


show_degradations(my_rgb, "your colormap")

# The real test: do far-apart values stay far apart for a deuteranopic reader?
sim = rgb_to_Jab(simulate_cvd(my_rgb, "deuteranopia"))
d = np.linalg.norm(sim[:, None, :] - sim[None, :, :], axis=-1)
far = np.abs(np.subtract.outer(np.arange(256), np.arange(256))) >= 52   # >= 20% apart
i, j = np.unravel_index(np.where(far, d, np.inf).argmin(), d.shape)
print(f"closest pair that is >=20% apart in the data:  positions {i/255:.0%} and {j/255:.0%}")
print(f"  they differ by only ΔE = {d[i, j]:.1f} for a deuteranopic reader "
      f"(under ~10 is a genuine problem)")

## 4. Compare against the benchmarks

`viridis` was built to be perceptually uniform. `jet` is the one this whole activity exists to
argue against — it is still the default in a lot of engineering software. Run this and look at
the smooth surface especially: `jet` puts sharp cyan and yellow bands into a surface that has no
edges in it at all.

`turbo` is Google's modern attempt to build a rainbow that does not lie, and it is the benchmark
Part 2 of the browser activity holds you against. Notice where it wins and where it does not: its
hue coverage and its gamut behavior are excellent, but its lightness is *not* monotonic — it runs
dark, bright, dark — so it still fails a grayscale printout. That is the compromise it chose, and
it is exactly the one you can beat.

`prism` is the pathological case. Watch its **wobble** number: at roughly 48% per step versus
viridis's 0.02%, its perceptual-deltas curve is pure noise, and the images it produces are
unreadable as quantities even though they contain every color.

In [ ]:
for name in ["viridis", "turbo", "jet", "prism"]:
    cmap = plt.get_cmap(name)
    rgb = cmap(np.linspace(0, 1, 256))[:, :3]
    show_on_data(cmap, name)
    diagnose(rgb, name)
    print()

In [ ]:
# Your colormap against both benchmarks, on the hardest of the three datasets.
z = DATASETS[0][1]
fig, axes = plt.subplots(1, 3, figsize=(11, 2.6))
for ax, (cmap, label) in zip(axes, [(my_cmap, "yours"), ("viridis", "viridis"), ("jet", "jet")]):
    ax.imshow(z, cmap=cmap)
    ax.set(title=label, xticks=[], yticks=[])
    ax.grid(False)
fig.suptitle("same surface, three colormaps", y=1.04)
plt.show()

## Using it for real

`my_cmap` is an ordinary matplotlib colormap, so it works anywhere one does:

```python
plt.imshow(Z, cmap=my_cmap); plt.colorbar()
plt.scatter(x, y, c=values, cmap=my_cmap)
sns.heatmap(df, cmap=my_cmap)
df.plot.hexbin("x", "y", cmap=my_cmap)
```

Reverse it with `my_cmap.reversed()`. Register it globally with
`matplotlib.colormaps.register(my_cmap)` and then refer to it by name.

## What to hand in

1. A screenshot of your score from the browser tool (aim for **80+**).
2. This notebook, run, with `MY_COLORS` set to your design.
3. Two or three sentences: which of the five criteria was hardest to satisfy, and what you had to
   give up to fix it? There is a real tension between chroma and uniformity, and between
   "pretty" and "honest" — say where you landed and why.

## The one thing to remember

A colormap is part of your analysis, not decoration on top of it. If the color scale is not
uniform, you have applied an invisible nonlinear transform to your data somewhere between the
computation and the reader's eye — and neither of you can see it happening.